<h1>Gold Bus Delays<h1>

<h3>Imports<h3>

In [ ]:
import pandas as pd
from sqlalchemy import create_engine
from config import DB_CONFIG
import psycopg2

<h3>Creating Gold Schema<h3>

In [14]:
conn = psycopg2.connect(**DB_CONFIG)
cur = conn.cursor()
cur.execute("CREATE SCHEMA IF NOT EXISTS gold;")
conn.commit()
conn.close()

<h2>Extracting data from Silver<h2>

In [12]:
engine = create_engine(f"postgresql://{DB_CONFIG['user']}:{DB_CONFIG['password']}@{DB_CONFIG['host']}:{DB_CONFIG['port']}/{DB_CONFIG['dbname']}")
df_matched = pd.read_sql("SELECT * FROM silver.bus_matched", engine)

df_matched.head()

,gps_id,gps_lat,gps_lon,time_gps,vehicle_number,schedule_id,trip_id,stop_id,route_id,route_name,stop_name,district,stop_sequence,stop_lat,stop_lon,arrival_time,departure_time,distance


<h3>Transforming data<h3>

In [ ]:
df_matched['delay_seconds'] =  (df_matched['arrival_time'] - df_matched['time_gps']).dt.total_seconds().astype(int)
df_matched['delay_minutes'] =   df_matched['delay_seconds'] / 60 
df_matched.round({'delay_minutes': 1})


KeyError: 'gps_time'

In [ ]:
df_gold = df_matched 
df_gold["gold_timestamp"] = pd.Timestamp.now()

<h2>Writing to DB<h2>

In [ ]:
df_gold.to_sql("bus_delays", engine, schema="gold", if_exists="append", index=False)